# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [31]:
# Write your code below.

%load_ext dotenv
%dotenv
%cd ./02_activities/assignments/
%run ../../01_materials/labs/update_path.py

#I specified the working directory from the root of the github repo to match the expectations for relative file paths in update_path.py

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file
/home/damian/Documents/machine_learning_fundamentals/02_activities/assignments


In [32]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [38]:
import os
from glob import glob

# Write your code below.
from utils.logger import get_logger
_logs = get_logger(__name__)

stock_files = glob(os.path.join(os.getenv("PRICE_DATA"),"**/*.parquet"), recursive = True)
_logs.info(f'Found {len(stock_files)} stock price files.')

2026-10-12 11:57:02,710, 3615713249.py, 9, INFO, Found 2132 stock price files.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
import pandas as pd #we'll need pandas for part of the transformation logic

#set index to ticker
dd_px = dd.read_parquet(stock_files).set_index("ticker")

#group by ticker to constrain shift calculations within tickers then get values lagging by 1 position
dd_shift = (
    dd_px
        .groupby('ticker', group_keys = False) #this groups to constrain the Date sorting and lag calculation
        .apply(
            lambda x: x.sort_values('Date', ascending = True) #this sorts by Date
                        .assign(Close_lag_1 = lambda df: df['Close'].shift(1)), #this creates a new "variable" to store the lagged value
            meta = dd_px._meta.assign(Close_lag_1 = pd.Series(dtype = 'float64')) #this will assign the new shifted series to the dataframe
    ))

#calculate returns
dd_feat = dd_shift.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low'] 
)

dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,returns,hi_lo_range
npartitions=60,,,,,,,,,,,,
ABM,datetime64[us],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64
ADSK,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
WWW,...,...,...,...,...,...,...,...,...,...,...,...
WWW,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

#compute the dd_rets dask dataframe to convert to a pandas dataframe
#dd_feat.compute()

### TROUBLESHOOTING: it looks like at least one of the partitions is missing ticker data ###
print("Expected columns:", dd_feat._meta.columns.tolist())
print("Expected dtypes:\n", dd_feat._meta.dtypes)

print("Dask columns:", dd_feat.columns.tolist())

#this is a lot of data so will take a while. There are 2132 partitions
dd_compute = dd_feat.get_partition(0).compute() #instantiate the dataframe
dd_compute["Rolling Return Mean"] = dd_compute["returns"].rolling(10).mean()
failed_partions = []
for i in range(1, dd_feat.npartitions):
    try:
        partition = dd_feat.get_partition(i).compute()
        partition["Rolling Return Mean"] = partition["returns"].rolling(10).mean()
        dd_compute = pd.concat([dd_compute, partition], axis = 0)
    except Exception as e:
        print(f"Partition {i} failed: {e}")
        failed_partions.append(i)

print("Failed partitions:")
print(failed_partions)

Expected columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'returns', 'hi_lo_range']
Expected dtypes:
 Date           datetime64[us]
Open                  float64
High                  float64
Low                   float64
Close                 float64
Adj Close             float64
Volume                float64
source                 string
Year                    int32
Close_lag_1           float64
returns               float64
hi_lo_range           float64
dtype: object
Dask columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'returns', 'hi_lo_range']


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,returns,hi_lo_range,Rolling Return Mean
ticker,,,,,,,,,,,,,
ABM,1980-03-17,0.000000,0.950000,0.933333,0.933333,0.027191,18400.0,ABM.csv,1980,NaN,NaN,0.016667,NaN
ABM,1980-03-18,0.000000,0.933333,0.908333,0.916667,0.026705,32800.0,ABM.csv,1980,0.933333,-0.017857,0.025000,NaN
ABM,1980-03-19,0.916667,0.916667,0.916667,0.916667,0.026705,0.0,ABM.csv,1980,0.916667,0.000000,0.000000,NaN
ABM,1980-03-20,0.000000,0.933333,0.925000,0.933333,0.027191,29600.0,ABM.csv,1980,0.916667,0.018182,0.008333,NaN
ABM,1980-03-21,0.000000,0.933333,0.933333,0.933333,0.027191,9600.0,ABM.csv,1980,0.933333,0.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
BCOM,2020-03-26,0.830000,0.910000,0.830000,0.910000,0.910000,7200.0,BCOM.csv,2020,0.820000,0.109756,0.080000,0.029381
BCOM,2020-03-27,0.790000,0.870000,0.790000,0.870000,0.870000,800.0,BCOM.csv,2020,0.910000,-0.043956,0.080000,0.020048
BCOM,2020-03-30,0.870000,0.890000,0.870000,0.890000,0.890000,2800.0,BCOM.csv,2020,0.870000,0.022988,0.020000,0.061170


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
"""
1. No, the conversion was not necessary. We could have created another Dask transformation for the  dataframe with returns and provide rolling mean instructions in dask.

2. I believe it would have been better in dask because it would have been run within the same instances as the `.compute()` function rather than run separately. Depending on how the data is read in and stored in memory, that may require less RAM.
"""

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.